## Cell 1: Setup
**What this demonstrates**: Environment initialisation for the adaptive routing pipeline — four strategies (naive, hybrid, corrective, multi-hop) and one classifier, all sharing the same clients and vector store.
**Expected output**: Setup confirmation with model names and the four query types.

In [ ]:
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv rank-bm25

import os
import json
import time
import pathlib
from dataclasses import dataclass, field
from typing import Literal
from collections import Counter

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# Two-model split: Haiku classifies and grades (fast); Sonnet generates (quality)
CLASSIFIER_MODEL = 'claude-haiku-4-5-20251001'
ANSWER_MODEL     = 'claude-sonnet-4-6'
EMBED_MODEL      = 'text-embedding-3-small'

# Four query types the classifier assigns — each routes to a different strategy
QueryType = Literal['simple_lookup', 'semantic_search', 'factual', 'multi_step']

client     = Anthropic()
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Classifier : {CLASSIFIER_MODEL}')
print(f'  Generator  : {ANSWER_MODEL}')
print(f'  Query types: simple_lookup | semantic_search | factual | multi_step')

## Cell 2: Data — Unified Knowledge Base
**What this demonstrates**: Loading all four sample documents into a single Chroma vector store and BM25 index — the unified corpus the adaptive router will query. Spanning lending policy, Basel III regulation, ISDA derivatives, and earnings data gives each query type a realistic retrieval challenge.
**Expected output**: Chunk count per document, total corpus size, and index confirmation.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

# All four sample documents — each represents a different knowledge domain
DOC_FILES = {
    'fintech_policy' : 'fintech_policy.txt',    # internal lending policy
    'basel_iii'      : 'basel_iii_excerpt.txt',  # regulatory capital framework
    'isda'           : 'isda_excerpt.txt',        # derivatives agreement
    'earnings_report': 'earnings_report.txt',     # quarterly financials
}

raw_docs: list[Document] = []
for source_name, filename in DOC_FILES.items():
    text = (BASE_DIR / filename).read_text(encoding='utf-8')
    raw_docs.append(Document(page_content=text, metadata={'source': source_name}))
    print(f'  {source_name:<18}: {len(text):,} chars')

# Chunk all documents with consistent settings
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks: list[Document] = splitter.split_documents(raw_docs)
# Stamp each chunk with an index for BM25 alignment
for i, c in enumerate(chunks):
    c.metadata['chunk_idx'] = i

# Dense index: Chroma for similarity search (semantic_search, factual, multi_step paths)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='adaptive_rag_corpus',
)

# Sparse index: BM25 for keyword retrieval (hybrid strategy)
tokenized_corpus = [c.page_content.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

print(f'\nCorpus : {len(chunks)} chunks across {len(raw_docs)} documents')
print('Indexes: Chroma (dense) + BM25 (sparse) ready')

## Cell 3: Core — Classifier, Strategies, Router
**What this demonstrates**: The full adaptive pipeline — `classify_query` assigns a type, four strategy functions implement the retrieval logic, and `adaptive_rag` routes each query to the right strategy and logs the decision.
**Expected output**: Function definitions confirmed — no output until Cell 4.

In [ ]:
@dataclass
class ClassificationResult:
    query_type: str   # simple_lookup | semantic_search | factual | multi_step
    rationale:  str   # classifier reasoning — logged for calibration

@dataclass
class RouteResult:
    query:         str
    query_type:    str
    rationale:     str
    strategy_used: str
    answer:        str
    retrieved:     list[str] = field(default_factory=list)
    latency_ms:    float = 0.0


# ── Classifier ────────────────────────────────────────────────────────────────

def classify_query(query: str) -> ClassificationResult:
    """Assign each incoming query a type using a few-shot prompted Haiku call.

    Types and routing targets:
        simple_lookup   → naive RAG       (definition, acronym, single known fact)
        semantic_search → hybrid RAG      (concept or theme requiring dense + sparse match)
        factual         → corrective RAG  (specific number, threshold, policy clause)
        multi_step      → multi-hop RAG   (cross-document comparison or synthesis)
    """
    prompt = (
        'Classify this query into exactly one type:\n\n'
        'simple_lookup   — definition, acronym, or single fact the model likely knows.\n'
        '                  e.g. "What does LCR stand for?", "Define Basel III."\n'
        'semantic_search — concept or theme needing semantic + keyword matching.\n'
        '                  e.g. "What are the key risk factors?", "Summarise the liquidity approach."\n'
        'factual         — specific verifiable number, threshold, or policy clause.\n'
        '                  e.g. "What is the minimum CET1 ratio?", "What is the LTV limit?"\n'
        'multi_step      — cross-document reasoning, comparison, or synthesis.\n'
        '                  e.g. "How does our policy compare to Basel III?", "What changed between Q1 and Q2?"\n\n'
        f'Query: {query}\n\n'
        'Return JSON only: {"type": "<one of the four>", "rationale": "<one sentence>"}'
    )
    response = client.messages.create(
        model=CLASSIFIER_MODEL,
        max_tokens=120,
        messages=[{'role': 'user', 'content': prompt}],
    )
    try:
        parsed = json.loads(response.content[0].text.strip())
        return ClassificationResult(query_type=parsed['type'], rationale=parsed['rationale'])
    except (json.JSONDecodeError, KeyError):
        # Fallback: default to semantic_search if classifier output is malformed
        return ClassificationResult(query_type='semantic_search', rationale='Classification parse failed — defaulting to semantic_search')


# ── Shared generation ─────────────────────────────────────────────────────────

def _generate(query: str, context: str) -> str:
    """Shared generation step used by all four strategies."""
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=350,
        system=(
            'You are a financial analyst. Answer using only the provided context. '
            'If the context does not contain the answer, say so explicitly.'
        ),
        messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'}],
    )
    return response.content[0].text.strip()


# ── Strategy 1: Naive RAG (simple_lookup) ─────────────────────────────────────

def strategy_naive_rag(query: str, k: int = 4) -> tuple[str, list[str]]:
    """simple_lookup: top-k similarity search, generate from retrieved context."""
    docs = vectorstore.similarity_search(query, k=k)
    context = '\n---\n'.join(d.page_content for d in docs)
    return _generate(query, context), [d.page_content[:80] for d in docs]


# ── Strategy 2: Hybrid RAG (semantic_search) ──────────────────────────────────

def strategy_hybrid_rag(query: str, k: int = 5) -> tuple[str, list[str]]:
    """semantic_search: BM25 + dense retrieval fused with RRF (k=60).

    RRF combines rank positions from both retrievers without requiring
    score normalisation — robust to score-scale differences between BM25 and cosine.
    """
    # BM25 candidates
    tokens = query.lower().split()
    bm25_scores = bm25.get_scores(tokens)
    top_bm25_idx = sorted(range(len(chunks)), key=lambda i: bm25_scores[i], reverse=True)[:k]
    bm25_docs = [chunks[i] for i in top_bm25_idx]

    # Dense candidates
    dense_docs = vectorstore.similarity_search(query, k=k)

    # RRF fusion: accumulate reciprocal rank scores per unique chunk
    rrf: dict[str, float] = {}
    doc_store: dict[str, Document] = {}
    for rank, doc in enumerate(bm25_docs):
        key = doc.page_content[:60]
        rrf[key] = rrf.get(key, 0.0) + 1 / (60 + rank)
        doc_store[key] = doc
    for rank, doc in enumerate(dense_docs):
        key = doc.page_content[:60]
        rrf[key] = rrf.get(key, 0.0) + 1 / (60 + rank)
        doc_store[key] = doc

    top_keys  = sorted(rrf, key=lambda kk: rrf[kk], reverse=True)[:k]
    fused     = [doc_store[kk] for kk in top_keys]
    context   = '\n---\n'.join(d.page_content for d in fused)
    return _generate(query, context), [f'[rrf={rrf[d.page_content[:60]]:.4f}] {d.page_content[:60]}' for d in fused]


# ── Strategy 3: Corrective RAG (factual) ──────────────────────────────────────

RELEVANCE_THRESHOLD = 3.0  # chunks below this score are discarded before generation

def strategy_corrective_rag(query: str, k: int = 5) -> tuple[str, list[str]]:
    """factual: retrieve → grade each chunk (1–5) → keep above threshold → generate.

    Ensures factual queries are answered only from chunks that actually contain
    the answer. Chunks below RELEVANCE_THRESHOLD are discarded. If no chunk
    passes, returns an explicit abstain rather than generating from poor context.
    """
    docs = vectorstore.similarity_search(query, k=k)

    graded: list[tuple[Document, int]] = []
    for doc in docs:
        grade_prompt = (
            f'Rate how relevant this excerpt is for answering the query on a scale 1–5.\n'
            f'5=directly answers with specific facts, 4=strongly relevant, '
            f'3=partially relevant, 2=low relevance, 1=not relevant.\n'
            f'Query: {query}\nExcerpt: {doc.page_content[:300]}\n'
            f'Return a single integer 1–5 only.'
        )
        resp = client.messages.create(
            model=CLASSIFIER_MODEL,  # Haiku: scoring task, not reasoning
            max_tokens=5,
            messages=[{'role': 'user', 'content': grade_prompt}],
        )
        try:
            score = int(resp.content[0].text.strip()[0])
        except (ValueError, IndexError):
            score = 3  # neutral default on parse failure
        graded.append((doc, score))

    passing = [(d, s) for d, s in graded if s >= RELEVANCE_THRESHOLD]
    if not passing:
        # No chunk met the quality bar — abstain rather than generate from poor context
        return 'Cannot answer with confidence — no sufficiently relevant chunks found.', []

    context = '\n---\n'.join(d.page_content for d, _ in passing)
    snippets = [f'[score={s}] {d.page_content[:70]}' for d, s in passing]
    return _generate(query, context), snippets


# ── Strategy 4: Multi-hop RAG (multi_step) ────────────────────────────────────

def strategy_multi_hop(query: str, k: int = 4) -> tuple[str, list[str]]:
    """multi_step: two-pass retrieval with an LLM-generated sub-question bridging the passes.

    Pass 1: retrieve on the original query.
    Bridge: LLM generates a follow-up sub-question based on the initial context.
    Pass 2: retrieve on the sub-question to fill the gap.
    Synthesis: generate a final answer from both passes combined.
    """
    # Pass 1
    docs_1 = vectorstore.similarity_search(query, k=k)
    context_1 = '\n---\n'.join(d.page_content for d in docs_1)

    # Bridge: generate a sub-question to guide pass 2
    sub_q_resp = client.messages.create(
        model=CLASSIFIER_MODEL,  # Haiku: sub-question generation is a focused task
        max_tokens=80,
        messages=[{'role': 'user', 'content': (
            f'Given this question and initial context, write one focused follow-up question '
            f'to retrieve the remaining information needed for a complete answer.\n\n'
            f'Question: {query}\n'
            f'Initial context:\n{context_1[:600]}\n\n'
            f'Follow-up question (one sentence only):'
        )}],
    )
    sub_question = sub_q_resp.content[0].text.strip()

    # Pass 2: retrieve on sub-question
    docs_2 = vectorstore.similarity_search(sub_question, k=k)
    context_2 = '\n---\n'.join(d.page_content for d in docs_2)

    # Synthesise from both passes
    combined = (
        f'[First retrieval — original query]\n{context_1}\n\n'
        f'[Second retrieval — sub-question: {sub_question}]\n{context_2}'
    )
    answer = _generate(query, combined)

    snippets = (
        [f'[P1] {d.page_content[:60]}' for d in docs_1] +
        [f'[P2 — sub-q] {d.page_content[:60]}' for d in docs_2]
    )
    return answer, snippets


# ── Router ────────────────────────────────────────────────────────────────────

STRATEGY_MAP: dict[str, tuple[str, object]] = {
    'simple_lookup' : ('Naive RAG',       strategy_naive_rag),
    'semantic_search': ('Hybrid RAG',     strategy_hybrid_rag),
    'factual'       : ('Corrective RAG',  strategy_corrective_rag),
    'multi_step'    : ('Multi-hop RAG',   strategy_multi_hop),
}

def adaptive_rag(query: str) -> RouteResult:
    """Classify → route → execute → log.

    The classifier runs first on every query. Its output determines which
    strategy fires. The routing decision and rationale are logged in the
    RouteResult for calibration monitoring.
    """
    t0 = time.perf_counter()

    cls = classify_query(query)
    strategy_label, strategy_fn = STRATEGY_MAP.get(
        cls.query_type, ('Naive RAG', strategy_naive_rag)  # safe fallback
    )
    answer, retrieved = strategy_fn(query)

    return RouteResult(
        query=query,
        query_type=cls.query_type,
        rationale=cls.rationale,
        strategy_used=strategy_label,
        answer=answer,
        retrieved=retrieved,
        latency_ms=(time.perf_counter() - t0) * 1000,
    )


print('Adaptive RAG pipeline defined')
print('  classify_query()          → ClassificationResult (type + rationale)')
print('  strategy_naive_rag()      → simple_lookup  path')
print('  strategy_hybrid_rag()     → semantic_search path')
print('  strategy_corrective_rag() → factual         path')
print('  strategy_multi_hop()      → multi_step      path')
print('  adaptive_rag()            → classify → route → execute → RouteResult')

## Cell 4: Run — Three Queries, Three Routing Decisions
**What this demonstrates**: Three queries each routed to a different strategy — the classifier's decision is printed before the answer, making the routing logic transparent.
**Expected output**: For each query: type, strategy selected, classifier rationale, latency, and answer.

In [ ]:
QUERIES = [
    'What does CET1 stand for in banking regulation?',
    'What are the key liquidity risk themes across our regulatory and policy documents?',
    'How do our internal loan-to-value limits compare to the Basel III capital buffer requirements?',
]

results: list[RouteResult] = []

for i, query in enumerate(QUERIES, 1):
    print('=' * 65)
    print(f'Query {i}: {query}')
    result = adaptive_rag(query)
    results.append(result)
    print(f'  Classified : {result.query_type}')
    print(f'  Strategy   : {result.strategy_used}')
    print(f'  Rationale  : {result.rationale}')
    print(f'  Latency    : {result.latency_ms:.0f} ms')
    print(f'\nAnswer:\n{result.answer}')
    print()

## Cell 5: Inspect — Classifier Reasoning and Strategy Selection
**What this demonstrates**: The routing log that monitors classifier behaviour — type distribution, per-query rationale, and retrieved chunk previews per strategy. In production this log is how you detect classifier drift.
**Expected output**: Routing log table, retrieved chunk previews annotated by strategy, and a routing distribution summary.

In [ ]:
# ── Routing log table ─────────────────────────────────────────────────────────
print('ROUTING LOG')
print(f"{'Query':<54} {'Type':<18} {'Strategy':<18} {'ms':>5}")
print('─' * 97)
for r in results:
    print(f'{r.query[:52]:<54} {r.query_type:<18} {r.strategy_used:<18} {r.latency_ms:>5.0f}')

# ── Per-query retrieval preview ───────────────────────────────────────────────
print('\n\nPER-QUERY RETRIEVAL DETAIL')
for r in results:
    print(f'\n── {r.query[:60]} ──')
    print(f'   Strategy  : {r.strategy_used}')
    print(f'   Classifier: {r.rationale}')
    if r.retrieved:
        print(f'   Chunks ({len(r.retrieved)}):')
        for chunk in r.retrieved[:3]:
            print(f'     · {chunk[:90]}')
    else:
        print('   No retrieval — abstained (no chunks passed threshold)')

# ── Routing distribution ──────────────────────────────────────────────────────
type_counts = Counter(r.query_type for r in results)
print('\n\nROUTING DISTRIBUTION (this session)')
print(f"{'Type':<18} {'Strategy':<18} {'Count'}")
print('─' * 42)
for qtype, count in sorted(type_counts.items()):
    strategy_label = STRATEGY_MAP.get(qtype, ('Unknown',))[0]
    print(f'{qtype:<18} {strategy_label:<18} {count}')
print()
print('In production: log this distribution weekly.')
print('A shift — e.g. multi_step rising from 20% to 50% — signals classifier drift')
print('or a change in query patterns that requires retraining the classifier.')

## Cell 6: Fintech — Unified Knowledge Base Q&A
**What this demonstrates**: A unified fintech assistant receiving queries of different complexity against the full corpus — policies, regulation, derivatives, and earnings data — with adaptive routing invisible to the caller.
**Expected output**: Routing decisions and grounded answers for three queries spanning the full complexity range, followed by a routing summary.

In [ ]:
FINTECH_QUERIES = [
    # Expected: simple_lookup → Naive RAG (definitional)
    'What is a Liquidity Coverage Ratio?',
    # Expected: factual → Corrective RAG (specific threshold from internal policy)
    'What is the maximum loan-to-value ratio specified in the lending policy?',
    # Expected: multi_step → Multi-hop RAG (cross-document comparison)
    'How do our internal liquidity requirements compare to the Basel III LCR framework, '
    'and what does the earnings report say about our current liquidity position?',
]

print('UNIFIED FINTECH KNOWLEDGE BASE — ADAPTIVE ROUTING')
print('Corpus: fintech_policy + basel_iii + isda + earnings_report')
print()

fintech_results: list[RouteResult] = []
for query in FINTECH_QUERIES:
    result = adaptive_rag(query)
    fintech_results.append(result)
    print('─' * 65)
    print(f'Query    : {query[:80]}')
    print(f'Routed to: {result.strategy_used}  [{result.query_type}]')
    print(f'Rationale: {result.rationale}')
    print(f'Latency  : {result.latency_ms:.0f} ms')
    print()
    answer_preview = result.answer[:350].replace('\n', '\n          ')
    print(f'Answer   : {answer_preview}')
    if len(result.answer) > 350:
        print('          ...')
    print()

# Routing summary
print('─' * 65)
print('ROUTING SUMMARY')
for r in fintech_results:
    print(f'  [{r.query_type:<18}] → {r.strategy_used:<18}  ({r.latency_ms:.0f} ms)')
print()
print('Each query received the strategy matched to its complexity.')
print('The caller saw one interface. The classifier handled the rest.')